In [14]:
# ==============================================================================
# Celda 1: Importación de librerías, configuración inicial y definición de rutas
# ==============================================================================

"""
Configuración inicial para la extracción del espacio fonético-fonológico.

Este script establece las dependencias y rutas necesarias para leer las 
transcripciones alineadas (TextGrids) del dataset ds003020, con el fin de 
construir las matrices de características de control para el modelado voxel-wise.
"""

import os
import glob
from pathlib import Path
from typing import List, Dict, Union

import numpy as np
import pandas as pd
import tgt  # Librería para parsear archivos Praat TextGrid

# Librerías sugeridas para la alineación temporal y convolución (si se elige esa opción)
import scipy.signal as signal
from scipy.interpolate import interp1d

# ------------------------------------------------------------------------------
# Configuración de Rutas (Paths)
# ------------------------------------------------------------------------------

# Ruta base donde se encuentran los TextGrids (según tu sistema en Ubuntu)
BASE_DIR = Path("/home/amont21/Documentos/voxelwise modeling/ds003020/derivatives/TextGrids")

def verificar_entorno(ruta_base: Path) -> None:
    """
    Verifica que el directorio de datos exista y lista los archivos disponibles.
    
    Args:
        ruta_base (Path): Objeto Path de pathlib que apunta al directorio de TextGrids.
        
    Raises:
        FileNotFoundError: Si el directorio especificado no existe en el sistema.
    """
    if not ruta_base.exists():
        raise FileNotFoundError(
            f"ERROR: No se encontró el directorio {ruta_base}. "
            "Por favor verifica la ruta o si el disco está montado."
        )
    
    # Buscar todos los archivos .TextGrid de manera recursiva
    archivos_tg = list(ruta_base.rglob("*.TextGrid"))
    
    print(f"Directorio base detectado exitosamente:\n{ruta_base}\n")
    print(f"Total de archivos .TextGrid encontrados: {len(archivos_tg)}")
    
    if archivos_tg:
        print("Ejemplo de archivos detectados:")
        for archivo in archivos_tg[:3]:  # Muestra solo los primeros 3
            print(f" - {archivo.name}")

# Ejecutar la verificación al iniciar el notebook
verificar_entorno(BASE_DIR)

# ------------------------------------------------------------------------------
# Parámetros Globales del Estudio (Extraídos de la Metodología)
# ------------------------------------------------------------------------------
TR_FMR = 2.0  # Tiempo de Repetición (Resolution temporal de la fMRI en segundos)
FRECUENCIA_MUESTREO_ALTA = 100  # Hz para la matriz temporal intermedia antes de bajar al TR

Directorio base detectado exitosamente:
/home/amont21/Documentos/voxelwise modeling/ds003020/derivatives/TextGrids

Total de archivos .TextGrid encontrados: 84
Ejemplo de archivos detectados:
 - odetostepfather.TextGrid
 - singlewomanseekingmanwich.TextGrid
 - mybackseatviewofagreatromance.TextGrid


In [15]:
# ==============================================================================
# Celda 2: Diccionario de Rasgos y Función de Extracción a Alta Resolución
# ==============================================================================

"""
Módulo para la extracción de características fonético-fonológicas.

Transforma las transcripciones fonéticas en ARPABET (extraídas de TextGrids)
en matrices temporales de alta resolución (100 Hz), utilizando una 
representación binaria de rasgos articulatorios y distintivos.
"""

# Definición del espacio de características (14 dimensiones)
# Cada fonema se representará como un vector binario [1, 0, ...]
NOMBRES_RASGOS = [
    'vocalico', 'consonantico', 
    'sordo', 'sonoro', 
    'bilabial', 'labiodental', 'dental_alveolar', 'palatal_velar', 
    'oclusivo', 'fricativo', 'africado', 'nasal', 'liquido', 'aproximante'
]

# Diccionario de mapeo: ARPABET -> Vector Binario de 14 Rasgos
# Nota: Esta es una simplificación funcional estándar para encoding models.
# Los sufijos críticos /t/, /d/, /s/, /z/ están perfectamente diferenciados.
ARPABET_MAP = {
    # Consonantes Oclusivas
    'P':  [0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'B':  [0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'T':  [0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0], # Sufijo Pasado Regular 1
    'D':  [0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0], # Sufijo Pasado Regular 2
    'K':  [0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    'G':  [0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    # Consonantes Fricativas
    'F':  [0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0],
    'V':  [0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0],
    'TH': [0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    'DH': [0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    'S':  [0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0], # Sufijo 3ra Persona 1
    'Z':  [0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0], # Sufijo 3ra Persona 2
    'SH': [0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
    'ZH': [0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
    'HH': [0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
    # Consonantes Africadas, Nasales, Líquidas y Aproximantes
    'CH': [0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
    'JH': [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
    'M':  [0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    'N':  [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0],
    'NG': [0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0],
    'L':  [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    'R':  [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    'Y':  [0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
    'W':  [0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1],
    # Las vocales se agrupan aquí bajo el rasgo [vocalico] por simplicidad del control
    # Si quisieras control vocálico estricto (alto, bajo, redondeado) se agregarían 3 columnas más.
}

def crear_matriz_fonetica(ruta_textgrid: Union[str, Path], fs: int = 100) -> pd.DataFrame:
    """
    Lee un archivo TextGrid y genera la matriz temporal de características fonéticas.
    
    Args:
        ruta_textgrid (Path o str): Ruta absoluta al archivo .TextGrid.
        fs (int): Frecuencia de muestreo de la matriz resultante (por defecto 100 Hz).
        
    Returns:
        pd.DataFrame: Matriz temporal donde el índice representa el tiempo 
                      en segundos y las columnas son los rasgos fonético-fonológicos.
                      Los valores son 1.0 (presencia) o 0.0 (ausencia).
    """
    # 1. Leer el archivo TextGrid
    tg = tgt.io.read_textgrid(str(ruta_textgrid), include_empty_intervals=True)
    
    # 2. Identificar la capa (tier) correspondiente a los fonemas.
    # En el dataset ds003020, las capas suelen llamarse 'phones' o 'phones '
    tier_fonemas = None
    for nombre_tier in tg.get_tier_names():
        if 'phone' in nombre_tier.lower():
            tier_fonemas = tg.get_tier_by_name(nombre_tier)
            break
            
    if tier_fonemas is None:
        raise ValueError(f"No se encontró la capa de fonemas en: {ruta_textgrid}")
    
    # 3. Determinar la duración total de la historia y crear la matriz vacía
    duracion_total = tier_fonemas.end_time
    total_muestras = int(np.ceil(duracion_total * fs))
    matriz_ceros = np.zeros((total_muestras, len(NOMBRES_RASGOS)))
    
    # 4. Proyectar los fonemas sobre la matriz temporal
    for intervalo in tier_fonemas.intervals:
        fonema_bruto = intervalo.text.strip().upper()
        
        # Limpiar números de estrés léxico si existen (ej. 'AH0' -> 'AH')
        fonema_limpio = ''.join([letra for letra in fonema_bruto if not letra.isdigit()])
        
        # Si es una vocal (no está en el mapa detallado, pero es vocal)
        if fonema_limpio in ['AA', 'AE', 'AH', 'AO', 'AW', 'AY', 'EH', 'ER', 'EY', 'IH', 'IY', 'OW', 'OY', 'UH', 'UW']:
            rasgos = [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] # [+vocalico, +sonoro, resto=0]
        else:
            # Obtener el vector de rasgos del diccionario (si es silencio '' o 'SP', retorna puros 0)
            rasgos = ARPABET_MAP.get(fonema_limpio, [0]*len(NOMBRES_RASGOS))
            
        # Calcular índices de inicio y fin en la matriz de alta resolución
        idx_inicio = int(np.floor(intervalo.start_time * fs))
        idx_fin = int(np.ceil(intervalo.end_time * fs))
        
        # Evitar desbordamientos de índice al final de la historia
        idx_fin = min(idx_fin, total_muestras)
        
        # Asignar los rasgos a esas muestras de tiempo (broadcasting)
        if sum(rasgos) > 0:  # Si no es un silencio
            matriz_ceros[idx_inicio:idx_fin, :] = rasgos

    # 5. Convertir a DataFrame para mejor manipulación y trazabilidad
    eje_temporal = np.arange(total_muestras) / fs
    df_caracteristicas = pd.DataFrame(matriz_ceros, columns=NOMBRES_RASGOS, index=eje_temporal)
    df_caracteristicas.index.name = 'Tiempo_Segundos'
    
    return df_caracteristicas

In [16]:
# ==============================================================================
# Celda 3: Auditoría empírica de los caracteres ARPABET en el corpus
# ==============================================================================

"""
Script de auditoría fonética.

Itera sobre todos los archivos TextGrid del directorio base para extraer 
el inventario completo y la frecuencia de los símbolos fonéticos (ARPABET) 
realmente presentes en los datos. Esto permite identificar sufijos de 
alineadores automáticos (ej. estrés, ruidos) para su posterior homogenización.
"""

from collections import Counter
import tgt

def auditar_inventario_arpabet(ruta_base: Path) -> pd.DataFrame:
    """
    Recorre los TextGrids y cuenta la frecuencia de todos los símbolos fonéticos.
    
    Args:
        ruta_base (Path): Ruta al directorio que contiene los .TextGrid.
        
    Returns:
        pd.DataFrame: Un dataframe con los símbolos encontrados y su frecuencia,
                      ordenado de mayor a menor frecuencia.
    """
    archivos_tg = list(ruta_base.rglob("*.TextGrid"))
    inventario_fonemas = Counter()
    archivos_leidos = 0
    archivos_con_error = []
    
    print(f"Iniciando auditoría sobre {len(archivos_tg)} archivos TextGrid...")
    
    for ruta_archivo in archivos_tg:
        try:
            # Leer el TextGrid
            tg = tgt.io.read_textgrid(str(ruta_archivo), include_empty_intervals=True)
            
            # Buscar la capa de fonemas (suele llamarse 'phones' o 'phones ')
            tier_fonemas = None
            for nombre_tier in tg.get_tier_names():
                if 'phone' in nombre_tier.lower():
                    tier_fonemas = tg.get_tier_by_name(nombre_tier)
                    break
            
            if tier_fonemas is None:
                archivos_con_error.append(f"{ruta_archivo.name} (Sin capa de fonemas)")
                continue
                
            # Contar cada token exactamente como aparece
            for intervalo in tier_fonemas.intervals:
                token = intervalo.text.strip()
                # Consideramos el string vacío como 'SILENCIO_VACIO' para que sea visible
                if token == "":
                    token = "<SILENCIO_VACIO>"
                inventario_fonemas[token] += 1
                
            archivos_leidos += 1
            
        except Exception as e:
            archivos_con_error.append(f"{ruta_archivo.name} (Error: {str(e)})")

    # Mostrar resumen de la auditoría
    print("-" * 50)
    print(f"Auditoría finalizada. Archivos procesados con éxito: {archivos_leidos}")
    if archivos_con_error:
        print(f"Archivos con problemas ({len(archivos_con_error)}):")
        for error in archivos_con_error[:5]: # Muestra los primeros 5 errores
            print(f" - {error}")
        if len(archivos_con_error) > 5:
            print("   ... (y otros más)")
    print("-" * 50)
    
    # Convertir el Counter a un DataFrame para visualizarlo bonito en Jupyter
    df_inventario = pd.DataFrame.from_dict(inventario_fonemas, orient='index', columns=['Frecuencia'])
    df_inventario.index.name = 'Simbolo_Original'
    df_inventario = df_inventario.sort_values(by='Frecuencia', ascending=False)
    
    return df_inventario

# Ejecutar la auditoría
df_arpabet_audit = auditar_inventario_arpabet(BASE_DIR)

# Mostrar los primeros 50 símbolos para analizar su estructura
print("Top 50 símbolos más frecuentes encontrados en los TextGrids:")
display(df_arpabet_audit.head(50))

# Si quieres ver TODOS los símbolos únicos, puedes usar:
# display(df_arpabet_audit)

Iniciando auditoría sobre 84 archivos TextGrid...
--------------------------------------------------
Auditoría finalizada. Archivos procesados con éxito: 82
Archivos con problemas (2):
 - legacy.TextGrid (Error: Invalid TextGrid header: "Praat chronological TextGrid text file"
0.0124716553288 819.988889088   ! Time domain.)
 - exorcism.TextGrid (Error: Invalid TextGrid header: "Praat chronological TextGrid text file"
0.0124716553288 955.021995465   ! Time domain.)
--------------------------------------------------
Top 50 símbolos más frecuentes encontrados en los TextGrids:


,Frecuencia
Simbolo_Original,
AH0,41458
N,36371
T,35682
D,24875
sp,23432
S,22134
L,18506
R,17805
AY1,17085


In [ ]:
# ==============================================================================
# Celda 4: Visualización completa y exportación del inventario fonético
# ==============================================================================

"""
Exporta el resultado de la auditoría a un archivo CSV para su revisión
detallada en editores externos (como VS Code) y ajusta las opciones 
de visualización de pandas para mostrar todos los registros en pantalla.
"""

import pandas as pd

# 1. Opción recomendada: Exportar a un archivo CSV
ruta_exportacion = "inventario_arpabet_completo.csv"
df_arpabet_audit.to_csv(ruta_exportacion)
print(f"✅ El inventario completo ha sido guardado exitosamente en:\n{ruta_exportacion}")
print("Puedes abrir este archivo directamente en VS Code.\n")

# 2. Mostrar la tabla completa en el Jupyter Notebook temporalmente
# Guardamos la configuración original
config_original = pd.get_option('display.max_rows')

# Le decimos a pandas que no limite la cantidad de filas
pd.set_option('display.max_rows', None)

print("-" * 50)
print("INVENTARIO COMPLETO (138 SÍMBOLOS):")
print("-" * 50)
display(df_arpabet_audit)

# Restauramos la configuración original para no afectar futuras tablas
pd.set_option('display.max_rows', config_original)

In [18]:
# ==============================================================================
# Celda 5: Homogenización Fonética y Extracción del Espacio de Características
# ==============================================================================

"""
Módulo para la extracción y homogenización de características fonético-fonológicas.

Limpia empíricamente los tokens ARPABET (eliminando marcadores de estrés y 
ruidos de alineación) y los mapea a un espacio de 14 dimensiones de rasgos 
articulatorios y distintivos, muestreado a 100 Hz.
"""

import re
import numpy as np
import pandas as pd
import tgt
from pathlib import Path
from typing import Union

# 1. Definición del Espacio de Características
NOMBRES_RASGOS = [
    'vocalico', 'consonantico', 
    'sordo', 'sonoro', 
    'bilabial', 'labiodental', 'dental_alveolar', 'palatal_velar', 
    'oclusivo', 'fricativo', 'africado', 'nasal', 'liquido', 'aproximante'
]

# 2. Diccionario base para Consonantes (Control de sufijos /t/, /d/, /s/, /z/)
ARPABET_CONSONANTES = {
    'P':  [0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'B':  [0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'T':  [0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0], 
    'D':  [0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0], 
    'K':  [0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    'G':  [0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    'F':  [0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0],
    'V':  [0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0],
    'TH': [0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    'DH': [0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    'S':  [0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0], 
    'Z':  [0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0], 
    'SH': [0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
    'ZH': [0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
    'HH': [0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
    'CH': [0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
    'JH': [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
    'M':  [0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    'N':  [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0],
    'NG': [0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0],
    'L':  [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    'R':  [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    'Y':  [0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
    'W':  [0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1]
}

# 3. Conjuntos estandarizados según la auditoría
VOCALES = {'AA', 'AE', 'AH', 'AO', 'AW', 'AY', 'EH', 'ER', 'EY', 'IH', 'IY', 'OW', 'OY', 'UH', 'UW'}

def procesar_token_arpabet(token_bruto: str) -> list:
    """
    Limpia un token bruto del TextGrid y lo mapea a sus rasgos articulatorios.
    
    Args:
        token_bruto (str): Token extraído directamente del archivo.
        
    Returns:
        list: Lista de 14 enteros (1s y 0s) representando los rasgos. 
              Devuelve una lista de ceros si es ruido, silencio o error.
    """
    # Limpieza inicial: Mayúsculas y quitar espacios
    token = str(token_bruto).strip().upper()
    
    # Expresión regular para quitar números (ej. 'AH0' -> 'AH') y caracteres especiales
    token_limpio = re.sub(r'[^A-Z]', '', token)
    
    vector_ceros = [0] * len(NOMBRES_RASGOS)
    
    # 1. Si después de limpiar quedó vacío, es silencio o ruido
    if not token_limpio:
        return vector_ceros
        
    # 2. Búsqueda en el diccionario de consonantes
    if token_limpio in ARPABET_CONSONANTES:
        return ARPABET_CONSONANTES[token_limpio]
        
    # 3. Búsqueda en el conjunto de vocales
    if token_limpio in VOCALES:
        # Asignamos [+vocalico, +sonoro], el resto en 0
        return [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
        
    # 4. Excepciones de letras solas detectadas en la auditoría
    # (El alineador a veces dejó 'A', 'E', 'O' como vocales genéricas)
    if token_limpio in {'A', 'E', 'I', 'O', 'U'}:
        return [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
        
    # 5. Si es ruido explícito de letras (BR, SP, LG, NS, etc.) u otro error (VIYO)
    return vector_ceros

def extraer_espacio_fonetico(ruta_textgrid: Union[str, Path], fs: int = 100) -> pd.DataFrame:
    """
    Genera la matriz temporal de características fonéticas para una historia.
    
    Args:
        ruta_textgrid (Path o str): Ruta absoluta al archivo .TextGrid.
        fs (int): Frecuencia de muestreo (100 Hz).
        
    Returns:
        pd.DataFrame: Matriz (tiempo x características), indexada en segundos.
    """
    tg = tgt.io.read_textgrid(str(ruta_textgrid), include_empty_intervals=True)
    
    tier_fonemas = None
    for nombre in tg.get_tier_names():
        if 'phone' in nombre.lower():
            tier_fonemas = tg.get_tier_by_name(nombre)
            break
            
    if tier_fonemas is None:
        raise ValueError(f"Sin capa fonética en {ruta_textgrid}")
        
    duracion = tier_fonemas.end_time
    total_muestras = int(np.ceil(duracion * fs))
    matriz = np.zeros((total_muestras, len(NOMBRES_RASGOS)), dtype=np.float32)
    
    for intervalo in tier_fonemas.intervals:
        rasgos = procesar_token_arpabet(intervalo.text)
        
        idx_inicio = int(np.floor(intervalo.start_time * fs))
        idx_fin = int(np.ceil(intervalo.end_time * fs))
        idx_fin = min(idx_fin, total_muestras)
        
        if sum(rasgos) > 0:
            matriz[idx_inicio:idx_fin, :] = rasgos

    tiempo = np.arange(total_muestras) / fs
    df = pd.DataFrame(matriz, columns=NOMBRES_RASGOS, index=tiempo)
    df.index.name = 'Tiempo_Segundos'
    
    return df

# ==============================================================================
# Prueba de la función con una historia válida del dataset
# ==============================================================================

# Vamos a buscar el primer TextGrid válido para probar que la matriz se crea bien
try:
    primer_archivo = next(BASE_DIR.rglob("*.TextGrid"))
    
    # Evitar probar con los dos archivos malos que encontramos antes
    while primer_archivo.name in ['legacy.TextGrid', 'exorcism.TextGrid']:
        primer_archivo = next(BASE_DIR.rglob("*.TextGrid"))

    print(f"Generando espacio fonético para: {primer_archivo.name} ...")
    df_fonetico = extraer_espacio_fonetico(primer_archivo)
    
    print(f"\n¡Éxito! Dimensiones de la matriz: {df_fonetico.shape} (Muestras x Rasgos)")
    print("Primeras 5 filas donde hay actividad fonética detectada:")
    
    # Mostramos un fragmento donde haya sonido (sum > 0)
    display(df_fonetico[(df_fonetico.sum(axis=1) > 0)].head(5))

except StopIteration:
    print("No se encontraron archivos para probar.")
except Exception as e:
    print(f"Error al generar la matriz: {e}")

Generando espacio fonético para: odetostepfather.TextGrid ...

¡Éxito! Dimensiones de la matriz: (82811, 14) (Muestras x Rasgos)
Primeras 5 filas donde hay actividad fonética detectada:


,vocalico,consonantico,sordo,sonoro,bilabial,labiodental,dental_alveolar,palatal_velar,oclusivo,fricativo,africado,nasal,liquido,aproximante
Tiempo_Segundos,,,,,,,,,,,,,,
0.03,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.04,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.05,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.06,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.07,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Refactorización

In [19]:
# ==============================================================================
# Cell 1: Setup, Paths, and Global Constants
# ==============================================================================

"""
Configuración inicial para el entorno de trabajo.
Define las dependencias, rutas de sistema y las constantes globales, 
incluyendo el control para exportar la auditoría y los parámetros de muestreo.
"""

import os
import re
from collections import Counter
from pathlib import Path
from typing import List, Dict, Union

import numpy as np
import pandas as pd
import tgt

# ------------------------------------------------------------------------------
# Rutas de sistema y configuración de auditoría
# ------------------------------------------------------------------------------
BASE_DIR = Path("/home/amont21/Documentos/voxelwise modeling/ds003020/derivatives/TextGrids")

# Interruptor lógico: Cambia a True cuando desees guardar el CSV en tu disco.
SAVE_CSV_AUDIT = False  
CSV_EXPORT_PATH = "audit_arpabet_inventory.csv"

# ------------------------------------------------------------------------------
# Parámetros temporales (Muestreo a alta resolución y fMRI)
# ------------------------------------------------------------------------------
# HIGH_RES_FS (100 Hz): Resolución temporal detallada (10 ms) para capturar 
# la duración exacta de los fonemas antes de simular la respuesta hemodinámica.
HIGH_RES_FS = 100  

# TR_FMRI (2.0 s): Tiempo de repetición del escáner (se usará en etapas futuras).
TR_FMRI = 2.0

In [20]:
# ==============================================================================
# Cell 2: Empirical Corpus Audit
# ==============================================================================

"""
Script de auditoría fonética. Iterará sobre los TextGrids para extraer el 
inventario real de símbolos (incluyendo los ruidos del alineador).
"""

def audit_arpabet_inventory(base_dir: Path, save_csv: bool, csv_path: str) -> pd.DataFrame:
    """
    Recorre los archivos TextGrid y cuenta las frecuencias de cada token.
    
    Args:
        base_dir (Path): Ruta al directorio de TextGrids.
        save_csv (bool): Determina si se exporta el resultado a un archivo.
        csv_path (str): Nombre/ruta del archivo CSV a guardar.
        
    Returns:
        pd.DataFrame: Símbolos encontrados ordenados por frecuencia.
    """
    textgrid_files = list(base_dir.rglob("*.TextGrid"))
    phone_inventory = Counter()
    
    for file_path in textgrid_files:
        try:
            tg = tgt.io.read_textgrid(str(file_path), include_empty_intervals=True)
            phone_tier = None
            
            for tier_name in tg.get_tier_names():
                if 'phone' in tier_name.lower():
                    phone_tier = tg.get_tier_by_name(tier_name)
                    break
                    
            if phone_tier is not None:
                for interval in phone_tier.intervals:
                    token = interval.text.strip()
                    if token == "":
                        token = "<EMPTY_SILENCE>"
                    phone_inventory[token] += 1
                    
        except Exception:
            # Errores de lectura (ej. formato 'chronological') se ignoran aquí
            pass

    # Convertir a DataFrame
    df_audit = pd.DataFrame.from_dict(
        phone_inventory, orient='index', columns=['frequency']
    )
    df_audit.index.name = 'original_symbol'
    df_audit = df_audit.sort_values(by='frequency', ascending=False)
    
    if save_csv:
        df_audit.to_csv(csv_path)
        print(f"Auditoría guardada exitosamente en: {csv_path}")
        
    return df_audit

# Ejecutar la auditoría (solo guardará el CSV si SAVE_CSV_AUDIT es True en Celda 1)
df_corpus_audit = audit_arpabet_inventory(BASE_DIR, SAVE_CSV_AUDIT, CSV_EXPORT_PATH)
print(f"Auditoría completa. Se encontraron {len(df_corpus_audit)} símbolos únicos.")

Auditoría completa. Se encontraron 138 símbolos únicos.


In [21]:
# ==============================================================================
# Cell 3: Feature Space Definition and Homogenization Mapping
# ==============================================================================

"""
Define el espacio de 14 dimensiones (rasgos articulatorios y distintivos) y 
la función de homogenización que limpia los sufijos numéricos y ruidos 
encontrados en la auditoría de la Celda 2.
"""

FEATURE_NAMES = [
    'vocalic', 'consonantal', 
    'voiceless', 'voiced', 
    'bilabial', 'labiodental', 'dental_alveolar', 'palatal_velar', 
    'stop', 'fricative', 'affricate', 'nasal', 'liquid', 'approximant'
]

# Mapa de consonantes ARPABET hacia el espacio de características (1 = Presente, 0 = Ausente)
CONSONANT_MAP = {
    'P':  [0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'B':  [0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    'T':  [0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0], # Sufijos pasados
    'D':  [0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0], # Sufijos pasados
    'K':  [0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    'G':  [0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    'F':  [0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0],
    'V':  [0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0],
    'TH': [0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    'DH': [0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    'S':  [0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0], # Sufijos 3ra persona
    'Z':  [0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0], # Sufijos 3ra persona
    'SH': [0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
    'ZH': [0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
    'HH': [0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
    'CH': [0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
    'JH': [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
    'M':  [0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    'N':  [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0],
    'NG': [0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0],
    'L':  [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    'R':  [0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    'Y':  [0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
    'W':  [0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1]
}

VOWEL_SET = {
    'AA', 'AE', 'AH', 'AO', 'AW', 'AY', 'EH', 'ER', 
    'EY', 'IH', 'IY', 'OW', 'OY', 'UH', 'UW'
}

def process_arpabet_token(raw_token: str) -> list:
    """
    Limpia un token ARPABET (remueve números y ruidos) y retorna sus rasgos.
    
    Args:
        raw_token (str): Símbolo bruto extraído del TextGrid.
        
    Returns:
        list: Vector de 14 enteros (ceros y unos) representando los rasgos.
    """
    token = str(raw_token).strip().upper()
    # Eliminar cualquier dígito u otro carácter no alfabético
    clean_token = re.sub(r'[^A-Z]', '', token)
    
    zero_vector = [0] * len(FEATURE_NAMES)
    
    if not clean_token:
        return zero_vector
        
    if clean_token in CONSONANT_MAP:
        return CONSONANT_MAP[clean_token]
        
    # Las excepciones vocálicas descubiertas en la auditoría ('A', 'E'...) se incluyen aquí
    if clean_token in VOWEL_SET or clean_token in {'A', 'E', 'I', 'O', 'U'}:
        # Retorna el patrón básico vocálico: [+vocalic, +voiced, el resto ceros]
        return [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
        
    # Para cualquier ruido restante detectado (SP, BR, LG), retorna silencio
    return zero_vector

In [22]:
# ==============================================================================
# Cell 4: High-Resolution Temporal Extraction Engine
# ==============================================================================

"""
Alinea temporalmente las características fonéticas. Convierte el TextGrid en 
una matriz a la resolución definida por HIGH_RES_FS (100 Hz), utilizando 
enteros de 8-bits para optimizar la memoria y visualización.
"""

def extract_phonetic_space(textgrid_path: Union[str, Path], fs: int) -> pd.DataFrame:
    """
    Genera la matriz temporal de características fonéticas.
    
    Args:
        textgrid_path (Path o str): Ruta absoluta al archivo .TextGrid.
        fs (int): Frecuencia de muestreo.
        
    Returns:
        pd.DataFrame: Matriz multidimensional de características.
    """
    tg = tgt.io.read_textgrid(str(textgrid_path), include_empty_intervals=True)
    
    phone_tier = None
    for name in tg.get_tier_names():
        if 'phone' in name.lower():
            phone_tier = tg.get_tier_by_name(name)
            break
            
    if phone_tier is None:
        raise ValueError(f"No phone tier found in {textgrid_path}")
        
    total_duration = phone_tier.end_time
    total_samples = int(np.ceil(total_duration * fs))
    
    # IMPORTANTE: dtype=np.int8 asegura que los valores sean enteros (0 y 1) 
    # y no decimales flotantes (0.0 y 1.0)
    feature_matrix = np.zeros((total_samples, len(FEATURE_NAMES)), dtype=np.int8)
    
    for interval in phone_tier.intervals:
        features = process_arpabet_token(interval.text)
        
        start_idx = int(np.floor(interval.start_time * fs))
        end_idx = int(np.ceil(interval.end_time * fs))
        end_idx = min(end_idx, total_samples)
        
        if sum(features) > 0:
            feature_matrix[start_idx:end_idx, :] = features

    time_axis = np.arange(total_samples) / fs
    
    df_features = pd.DataFrame(
        feature_matrix, 
        columns=FEATURE_NAMES, 
        index=time_axis
    )
    df_features.index.name = 'time_seconds'
    
    return df_features

In [25]:
# ==============================================================================
# Cell 5: Execution and Feature Space Visualization
# ==============================================================================

"""
Prueba el motor de extracción tomando la primera historia válida del corpus 
y despliega en pantalla un fragmento de la matriz resultante para confirmar 
que el espacio de características se construyó correctamente (enteros 0 y 1).
"""

try:
    # Buscar el primer archivo válido (evitando los formatos problemáticos)
    textgrid_files = list(BASE_DIR.rglob("*.TextGrid"))
    valid_file = None
    
    for file_path in textgrid_files:
        if file_path.name not in ['legacy.TextGrid', 'exorcism.TextGrid']:
            valid_file = file_path
            break
            
    if valid_file:
        print(f"Extrayendo espacio de características fonéticas para: {valid_file.name}")
        
        # Ejecutar función de extracción
        df_phonetic_space = extract_phonetic_space(valid_file, HIGH_RES_FS)
        
        print("\nInformación del Espacio de Características:")
        print(f"  -> Dimensiones (Muestras Temporales x Rasgos): {df_phonetic_space.shape}")
        print(f"  -> Tipo de dato: {df_phonetic_space.values.dtype}\n")
        
        print("Visualización de las primeras 10 muestras con actividad fonética:")
        # Filtramos para mostrar solo momentos donde hay sonido (sum > 0)
        active_samples = df_phonetic_space[(df_phonetic_space.sum(axis=1) > 0)]
        display(active_samples.head(15))
        
    else:
        print("Error: No se encontró ningún archivo válido para procesar.")

except Exception as e:
    print(f"Ocurrió un error en la ejecución: {str(e)}")

Extrayendo espacio de características fonéticas para: odetostepfather.TextGrid

Información del Espacio de Características:
  -> Dimensiones (Muestras Temporales x Rasgos): (82811, 14)
  -> Tipo de dato: int8

Visualización de las primeras 10 muestras con actividad fonética:


,vocalic,consonantal,voiceless,voiced,bilabial,labiodental,dental_alveolar,palatal_velar,stop,fricative,affricate,nasal,liquid,approximant
time_seconds,,,,,,,,,,,,,,
0.03,1,0,0,1,0,0,0,0,0,0,0,0,0,0
0.04,1,0,0,1,0,0,0,0,0,0,0,0,0,0
0.05,1,0,0,1,0,0,0,0,0,0,0,0,0,0
0.06,1,0,0,1,0,0,0,0,0,0,0,0,0,0
0.07,1,0,0,1,0,0,0,0,0,0,0,0,0,0
0.08,1,0,0,1,0,0,0,0,0,0,0,0,0,0
0.09,1,0,0,1,0,0,0,0,0,0,0,0,0,0
0.10,1,0,0,1,0,0,0,0,0,0,0,0,0,0
0.11,1,0,0,1,0,0,0,0,0,0,0,0,0,0
